In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import numpy as np
from PIL import Image
from collections import Counter
import os

In [ ]:
VEHICLE_CLASSES = ["car", "bus", "truck"]
DATASET_DIR = "../data/coco"
SPLIT = "validation"

## Download COCO val2017 — vehicle classes only

In [ ]:
dataset = foz.load_zoo_dataset(
    "coco-2017",
    split=SPLIT,
    label_types=["detections"],
    classes=VEHICLE_CLASSES,
    only_matching=True,
    dataset_dir=DATASET_DIR,
)

print(f"Total samples: {len(dataset)}")

## Class distribution

In [ ]:
class_counts = Counter()
images_per_class = Counter()

for sample in dataset:
    labels_in_sample = set()
    if sample.ground_truth:
        for det in sample.ground_truth.detections:
            class_counts[det.label] += 1
            labels_in_sample.add(det.label)
    for label in labels_in_sample:
        images_per_class[label] += 1

df = pd.DataFrame({
    "class": list(class_counts.keys()),
    "total_instances": list(class_counts.values()),
    "images_containing": [images_per_class[c] for c in class_counts.keys()]
}).sort_values("total_instances", ascending=False)

print(df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(df["class"], df["total_instances"], color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title("Total instances per class")
axes[0].set_ylabel("Count")

axes[1].bar(df["class"], df["images_containing"], color=["#4C72B0", "#DD8452", "#55A868"])
axes[1].set_title("Images containing each class")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("../data/coco/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Bounding box size distribution

In [ ]:
bbox_areas = {cls: [] for cls in VEHICLE_CLASSES}

for sample in dataset:
    if not sample.ground_truth:
        continue
    img = Image.open(sample.filepath)
    w, h = img.size
    for det in sample.ground_truth.detections:
        bw = det.bounding_box[2] * w
        bh = det.bounding_box[3] * h
        bbox_areas[det.label].append(bw * bh)

fig, ax = plt.subplots(figsize=(10, 5))
for cls, areas in bbox_areas.items():
    ax.hist(np.sqrt(areas), bins=50, alpha=0.6, label=cls)
ax.set_xlabel("Bounding box side length (sqrt of area, px)")
ax.set_ylabel("Count")
ax.set_title("Bounding box size distribution per class")
ax.legend()
plt.tight_layout()
plt.savefig("../data/coco/bbox_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Sample images with ground truth boxes

In [ ]:
CLASS_COLORS = {"car": "#4C72B0", "bus": "#DD8452", "truck": "#55A868"}

samples = list(dataset.take(9))

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ax, sample in zip(axes.flatten(), samples):
    img = Image.open(sample.filepath)
    w, h = img.size
    ax.imshow(img)
    if sample.ground_truth:
        for det in sample.ground_truth.detections:
            x, y, bw, bh = det.bounding_box
            rect = patches.Rectangle(
                (x * w, y * h), bw * w, bh * h,
                linewidth=2, edgecolor=CLASS_COLORS.get(det.label, "red"), facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(x * w, y * h - 4, det.label, color=CLASS_COLORS.get(det.label, "red"),
                    fontsize=8, fontweight="bold")
    ax.axis("off")

plt.suptitle("COCO val2017 — vehicle samples with ground truth", fontsize=14)
plt.tight_layout()
plt.savefig("../data/coco/sample_images.png", dpi=150, bbox_inches="tight")
plt.show()

## Export image paths + annotations for evaluation notebooks

In [ ]:
records = []
for sample in dataset:
    if not sample.ground_truth:
        continue
    img = Image.open(sample.filepath)
    w, h = img.size
    for det in sample.ground_truth.detections:
        x, y, bw, bh = det.bounding_box
        records.append({
            "image_id": sample.id,
            "filepath": sample.filepath,
            "label": det.label,
            "x1": x * w,
            "y1": y * h,
            "x2": (x + bw) * w,
            "y2": (y + bh) * h,
            "img_width": w,
            "img_height": h,
        })

annotations_df = pd.DataFrame(records)
annotations_df.to_csv("../data/coco/annotations.csv", index=False)
print(f"Saved {len(annotations_df)} annotations from {annotations_df['image_id'].nunique()} images")
annotations_df.head()